# CDC Seasonal Vaccine Prediction Model

The Goal of this Program is to try and predict the vaccination rate of a given county using a machine learning model. 

This is done using data provided from the CDC. Follow the steps to generate and run the model.

Visit the CDC Website and preview the dataset for "Influenza Vaccination Coverage for All Ages (6+ Months)" here:

https://data.cdc.gov/Flu-Vaccinations/Influenza-Vaccination-Coverage-for-All-Ages-6-Mont/vh55-3he6/about_data

Once there, Select "Export" on the right side of the page and download the dataset as a CSV file, leaving the file in your downloads folder

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

#User input filepath
file_path = input("Copy and Paste the file path here: ")

#Remove accidental quotes if the user pasted "C:\path\to\file.csv"
file_path = file_path.strip().strip('"').strip("'")

#Normalize or replace extra slashes if needed
file_path = file_path.replace("\\\\", "\\")

#Convert to a Path object and resolve issues like '~' home shortcuts
file_path = Path(file_path).expanduser().resolve()

#Check if the path actually exists and is a file
if file_path.is_file():
    #Print success if it is
    print(f"Success! Found file: {file_path.name}")
else:
    #print Error if it isn't
   print(f"Error: The path '{file_path}' does not point to a valid file.")

Copy and Paste the file path here:  "C:\Users\yousu\Downloads\Influenza_Vaccination_Coverage_for_All_Ages_(6+_Months)_20261020.csv"


Success! Found file: Influenza_Vaccination_Coverage_for_All_Ages_(6+_Months)_20261020.csv


In [2]:
#Preprocessing
#Inport the CSV into Python as a dataframe
df = pd.read_csv(file_path)

#We are only interested in the Seasonal Vaccine so this filters the data to only the Seasonal Influenza Vaccine
df = df[df['Vaccine'] == 'Seasonal Influenza']

#We are only interested in counties so this statement filters out states/etc and leaves only counties
df = df[df['Geography Type'] == 'Counties']

#After filtering the above are 2 dimensions that are left in the Dataframe at this point for Dimension Type; "Age" and ">=18 Years". We want Age
df = df[df['Dimension Type'] == 'Age']

#There are counties that have the same name but are in different states. Concatenating County Name with FIPS yields a unique County Categorical field
df['County'] = df['Geography'].astype(str)+df['FIPS'].astype(str)

#Here there are a few features that have been exluded due to the fact that they are all the same value; Month(For counties, all months are '1'), Vaccine(we are testing only the Seasonal Influenza Vaccine), Dimension (The dimensions are all the same for counties (Age6+), and Sample Size(NA for Counties) exist but offer nothing to the model
df = df[['Season/Survey Year','County','Estimate (%)']]

#Need to cast "Season/Survey Year" as an integer to make it easier to filter later on
df['Season/Survey Year'] = df['Season/Survey Year'].astype(int)

#Convert Estimate to a number data type (it is object currently)
df['Estimate (%)'] = df['Estimate (%)'].astype(float)

# Convert County to a category type for our model later
df['County'] = df['County'].astype('category')

# Sort by year to ensure correct temporal ordering
df = df.sort_values(by='Season/Survey Year').reset_index(drop=True)

In [3]:
#Need to get the latest year +1 as our predict
last_year = int(df['Season/Survey Year'].max())
future_year = last_year + 1

In [4]:
# To capture true time-series dependency, we track what the percentage was last year and add it to our Dataframe as a new field
df['Estimate (%) Lag'] = df.groupby('County', observed = False)['Estimate (%)'].shift(1)

# Drop the first year since it won't have a lag value
df = df.dropna().reset_index(drop=True)

In [5]:
#Define the Categorical and Ordinal fields for the pipelies
numeric_columns = ['Estimate (%) Lag']
categorical_columns = ['County']
ordinal_columns = ['Season/Survey Year']

In [6]:
# We train on years up to the last, and test on the last year
train_set= df[df['Season/Survey Year'] < last_year]
test_set= df[df['Season/Survey Year'] >= last_year]

#Create Train set
y_train = train_set['Estimate (%)']
y_train = y_train.reset_index(drop=True)

X_train = train_set.drop(['Estimate (%)'], axis = 1)
X_train = X_train.reset_index(drop=True)

#Create Test Set
y_test = test_set['Estimate (%)']
y_test = y_test.reset_index(drop=True)

X_test = test_set.drop(['Estimate (%)'], axis = 1)
X_test = X_test.reset_index(drop=True)

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

#Pipeline for Numeric Columns
numeric_pipeline = Pipeline([
        ('imputer', SimpleImputer(strategy="median").set_output(transform="pandas")), 
        ('std_scaler', StandardScaler()), 
    ])

In [8]:
from sklearn.preprocessing import OneHotEncoder

#Create the pipeline for categorical columns
categorical_pipeline = Pipeline([
        ('Imputer', SimpleImputer(strategy = "most_frequent").set_output(transform='pandas')),
        ('Onehotencoder', OneHotEncoder(drop='first',handle_unknown='ignore')),
         ])

In [9]:
from sklearn.preprocessing import OrdinalEncoder

#Create the pipeline for ordinal columns
ordinal_pipeline = Pipeline([
        ('Imputer', SimpleImputer(strategy = "median").set_output(transform='pandas')),
        ('Ordinalencoder', OrdinalEncoder()),
         ])

In [10]:
from sklearn.compose import ColumnTransformer

#Create the column transformer
column_transformer = ColumnTransformer([
    ('num',numeric_pipeline, numeric_columns),
    ('cat',categorical_pipeline, categorical_columns),
    ('ord',ordinal_pipeline, ordinal_columns),
])    

In [11]:
#Run the train data through the transformer
X_train_transformed = column_transformer.fit_transform(X_train)

In [12]:
#Linear Regression
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()

#Linear model
lin_reg_model = lin_reg.fit(X_train_transformed, y_train)

In [13]:
# SVM Regression
from sklearn.svm import SVR

svm_reg = SVR()

#SVM model
svm_model = svm_reg.fit(X_train_transformed, y_train)

In [14]:
#Lasso Regression
from sklearn.linear_model import Lasso

lasso_reg = Lasso()

#Lasso model
lasso_model = lasso_reg.fit(X_train_transformed, y_train)

In [15]:
#Cross Validation Score
from sklearn.model_selection import cross_val_score

#CV Lin Model
cv_scores_lin_reg_model = -cross_val_score(lin_reg_model, X_train_transformed, y_train,
                                scoring="neg_root_mean_squared_error", cv=5)
#CV SVM model
cv_scores_svm_model = -cross_val_score(svm_model, X_train_transformed, y_train,
                                scoring="neg_root_mean_squared_error", cv=5)
#CV Lasso model
cv_scores_lasso_model = -cross_val_score(lasso_model, X_train_transformed, y_train,
                                scoring="neg_root_mean_squared_error", cv=5)

In [16]:
#RMSE Scores

rmse_lin_reg_model = np.mean(cv_scores_lin_reg_model)
rmse_svm_model = np.mean(cv_scores_svm_model)
rmse_lasso_model = np.mean(cv_scores_lasso_model)

print(
rmse_lin_reg_model,
rmse_svm_model,
rmse_lasso_model
)

3.7817089100454853 3.6144967970128237 4.520777208783412
